# DeepSeek-R1-0528-Qwen3-8BをGoogle Colabで動かす（4bit + Gradio）

添付の **Phi-4 Colabチャットノートブック**と同じ流れで、DeepSeekの公式小型Reasoningモデルを4bit量子化して動かします。

- 使用モデル: `deepseek-ai/DeepSeek-R1-0528-Qwen3-8B`
- 量子化: bitsandbytes NF4 4bit（読み込み時量子化）
- 推奨GPU: **L4 / A100**。T4でも短いコンテキストなら試せます。
- UI: Gradio
- Google DriveへのHugging Faceキャッシュ保存を選択可能

> **なぜDeepSeek-V4-Flash-0731本体ではないのか**  
> 2026-08時点のV4-Flash-0731公式チェックポイントは約167GBで、極低bit GGUFでも約80GB超です。単一Colab GPUに常駐させるPhi-4型の実行には大きすぎるため、このノートでは「実際にColabで扱える公式DeepSeekモデル」を優先します。

> DeepSeek-R1-0528ではsystem promptがサポートされ、公式のWeb/Appではtemperature 0.6が使われています。このノートもそれに寄せています。


In [1]:
# =========================================
# コード1 実行環境の準備
# =========================================
!nvidia-smi -L || echo "No GPU"
!python -V

# TorchとGradioはColab既定版をできるだけ利用する。
# Qwen3対応Transformersと量子化関連のみ更新する。
%pip -q install -U "transformers>=4.52,<5" accelerate bitsandbytes

import torch
import transformers

if not torch.cuda.is_available():
    raise RuntimeError("Colabの『ランタイム > ランタイムのタイプを変更』でGPUを有効にしてください。")

print("GPU:", torch.cuda.get_device_name(0))
print("CUDA capability:", torch.cuda.get_device_capability(0))
print("transformers:", transformers.__version__)
print("bf16 supported:", torch.cuda.is_bf16_supported())


GPU 0: NVIDIA RTX PRO 6000 Blackwell Server Edition (UUID: GPU-8642ccee-4a9b-8e52-972d-cb30052f59ea)
Python 3.12.13
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 272.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 88.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 74.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
CUDA capability: (12, 0)
transformers: 4.57.6
bf16 supported: True


In [2]:
# =========================================
# コード2 Google Driveとキャッシュ設定
# =========================================
from google.colab import drive
from pathlib import Path
import os
import shutil

USE_DRIVE_CACHE = True   # Driveに保存しない場合はFalse

if USE_DRIVE_CACHE:
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/Colab Notebooks/LocalLLM")
    CACHE_DIR = PROJECT_DIR / "Program" / "hf_cache"
else:
    CACHE_DIR = Path("/content/hf_cache")

CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ["HF_HUB_CACHE"] = str(CACHE_DIR)

usage = shutil.disk_usage(CACHE_DIR)
print("CACHE_DIR:", CACHE_DIR)
print(f"free space: {usage.free / 1024**3:.1f} GB")

# 元のBF16重みをダウンロードしてから4bitで読み込むため、
# 余裕を見て20GB以上の空きを推奨する。
if usage.free < 20 * 1024**3:
    print("WARNING: キャッシュの空きが20GB未満です。USE_DRIVE_CACHE=Falseも検討してください。")


Mounted at /content/drive
CACHE_DIR: /content/drive/MyDrive/Colab Notebooks/LocalLLM/Program/hf_cache
free space: 179.1 GB


In [3]:
# =========================================
# コード3 DeepSeek-R1-0528-Qwen3-8Bを4bitで読み込む
# =========================================
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "deepseek-ai/DeepSeek-R1-0528-Qwen3-8B"

COMPUTE_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    use_fast=True,
    cache_dir=str(CACHE_DIR),
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    cache_dir=str(CACHE_DIR),
    low_cpu_mem_usage=True,
)
model.eval()

INPUT_DEVICE = model.get_input_embeddings().weight.device

print("model loaded:", MODEL_ID)
print("input device:", INPUT_DEVICE)
print("4bit:", getattr(model, "is_loaded_in_4bit", False))
print("GPU allocated: %.2f GB" % (torch.cuda.memory_allocated() / 1024**3))


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/859 [00:00<?, ?B/s]

Unrecognized keys in `rope_scaling` for 'rope_type'='yarn': {'attn_factor'}


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-000002.safetensors:   0%|          | 0.00/7.77G [00:00<?, ?B/s]

model-00001-of-000002.safetensors:   0%|          | 0.00/8.61G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

model loaded: deepseek-ai/DeepSeek-R1-0528-Qwen3-8B
input device: cuda:0
4bit: True
GPU allocated: 5.66 GB


In [4]:
# =========================================
# コード4 応答生成関数
# =========================================
import torch
from datetime import datetime


def split_reasoning(text: str):
    """DeepSeek系の<think>...</think>があれば reasoning / answer に分ける。"""
    text = (text or "").strip()
    if "</think>" in text:
        left, answer = text.split("</think>", 1)
        reasoning = left.replace("<think>", "", 1).strip()
        return reasoning, answer.strip()
    return "", text


@torch.inference_mode()
def chat_generate(
    messages,
    max_new_tokens=1024,
    show_reasoning=False,
    temperature=0.6,
    top_p=0.95,
):
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    ).to(INPUT_DEVICE)

    outputs = model.generate(
        **inputs,
        max_new_tokens=int(max_new_tokens),
        do_sample=True,
        temperature=float(temperature),
        top_p=float(top_p),
        repetition_penalty=1.03,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
        use_cache=True,
    )

    generated_ids = outputs[0, inputs["input_ids"].shape[-1]:]
    raw = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    reasoning, answer = split_reasoning(raw)

    if show_reasoning and reasoning:
        return f"### Thinking\n{reasoning}\n\n### Answer\n{answer}"
    return answer or raw


def make_system_prompt():
    today = datetime.now().strftime("%Y-%m-%d")
    return (
        "あなたはDeepSeek-R1を用いた親切なAIアシスタントです。"
        "ユーザーと同じ言語で、正確かつ分かりやすく答えてください。"
        f"今日は{today}です。"
    )


In [5]:
# =========================================
# コード5 動作確認
# =========================================
messages = [
    {"role": "system", "content": make_system_prompt()},
    {"role": "user", "content": "日本語で1文だけ自己紹介してください。"},
]

print(chat_generate(messages, max_new_tokens=512, show_reasoning=False))


こんにちは、私はDeepSeek-R1です。日本語を話すことができます。あなたのAIアシスタントとして親切に、正確にお手伝いします。


In [6]:
# =========================================
# コード6 Gradioを用いたDeepSeekチャットUI
# =========================================
import gradio as gr
import inspect


def make_messages_chatbot(**kwargs):
    if "type" in inspect.signature(gr.Chatbot).parameters:
        kwargs["type"] = "messages"
    return gr.Chatbot(**kwargs)


def gr_chat(history, user_msg, show_reasoning):
    history = history or []
    user_msg = (user_msg or "").strip()

    if not user_msg:
        return history, "", history

    messages = [
        {"role": "system", "content": make_system_prompt()},
        *history[-8:],
        {"role": "user", "content": user_msg},
    ]

    reply = chat_generate(
        messages,
        max_new_tokens=1024,
        show_reasoning=bool(show_reasoning),
        temperature=0.6,
        top_p=0.95,
    )

    new_history = history + [
        {"role": "user", "content": user_msg},
        {"role": "assistant", "content": reply},
    ]
    return new_history, "", new_history


with gr.Blocks(title="DeepSeek-R1-0528-Qwen3-8B 4bit Chat") as chat_demo:
    gr.Markdown(
        "## DeepSeek-R1-0528-Qwen3-8B — Local 4bit Chat\n"
        "Colab上で重みをローカル推論します。外部APIは使いません。"
    )

    chatbot = make_messages_chatbot(
        label="Chat",
        show_label=False,
        sanitize_html=True,
    )
    chat_state = gr.State([])

    user_box = gr.Textbox(
        placeholder="質問を入力してください。",
        label="",
        lines=3,
    )
    show_reasoning = gr.Checkbox(
        value=False,
        label="Thinkingも表示する",
    )

    with gr.Row():
        send_btn = gr.Button("Send", variant="primary")
        clear_btn = gr.Button("Clear")

    send_btn.click(
        gr_chat,
        inputs=[chat_state, user_box, show_reasoning],
        outputs=[chat_state, user_box, chatbot],
        queue=False,
    )
    user_box.submit(
        gr_chat,
        inputs=[chat_state, user_box, show_reasoning],
        outputs=[chat_state, user_box, chatbot],
        queue=False,
    )
    clear_btn.click(
        lambda: ([], "", []),
        outputs=[chat_state, user_box, chatbot],
    )

print("WARNING: share=Trueで一時的な公開URLが作成されます。")
chat_demo.launch(
    share=True,
    inline=True,
    debug=False,
)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a639f95014bf5f6ce3.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## メモ

- 長いThinkingを生成すると、8Bでも応答に時間がかかります。まずは `max_new_tokens=1024` 程度で試してください。
- L4でメモリ不足になる場合は、会話履歴 `history[-8:]` を `history[-4:]` に減らすか、生成長を短くします。
- このノートはDeepSeek APIを使わず、Colabにダウンロードしたモデルをローカル推論します。
